# Mission 16 - 기본 미션: MNIST CNN 학습 + 모델 포맷 변환

**목표**:
1. MNIST CNN 모델 학습 (PyTorch)
2. 모델 3종 저장: `.pth` (일반), `.pth` (동적 양자화), `.onnx`
3. 모델 크기 비교

In [ ]:
# 필요 라이브러리 설치 (Colab 환경)
# !pip install torch torchvision onnx onnxruntime

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

PyTorch version: 2.10.0
Using device: mps


## 1. 데이터 로드

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data/mnist_data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data/mnist_data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset):,}장, Test: {len(test_dataset):,}장")

Train: 60,000장, Test: 10,000장


## 2. 모델 정의

In [3]:
class MnistCNN(nn.Module):
    """MNIST 분류용 CNN 모델 (LeNet 스타일).
    
    입력: (N, 1, 28, 28) float32
    출력: (N, 10) logits
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # 28x28 → 28x28
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 14x14 → 14x14
        self.pool = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.relu(self.conv1(x)))  # → (N, 32, 14, 14)
        x = self.pool(self.relu(self.conv2(x)))  # → (N, 64, 7, 7)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)               # → (N, 3136)
        x = self.relu(self.fc1(x))              # → (N, 128)
        x = self.fc2(x)                         # → (N, 10)
        return x

model = MnistCNN().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n총 파라미터 수: {total_params:,}")

MnistCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (dropout): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

총 파라미터 수: 421,642


## 3. 학습

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

Epoch 1/5 | Train Loss: 0.1362, Acc: 0.9586 | Val Loss: 0.0428, Acc: 0.9846
Epoch 2/5 | Train Loss: 0.0477, Acc: 0.9853 | Val Loss: 0.0372, Acc: 0.9878
Epoch 3/5 | Train Loss: 0.0357, Acc: 0.9883 | Val Loss: 0.0338, Acc: 0.9884
Epoch 4/5 | Train Loss: 0.0255, Acc: 0.9911 | Val Loss: 0.0304, Acc: 0.9898
Epoch 5/5 | Train Loss: 0.0216, Acc: 0.9934 | Val Loss: 0.0278, Acc: 0.9909


## 4. 모델 저장 - 3종 포맷

In [5]:
os.makedirs('./data/models', exist_ok=True)

# 4-1. 일반 PyTorch 모델 저장
pth_path = './data/models/mission_16_mnist_cnn.pth'
torch.save(model.state_dict(), pth_path)
print(f"[저장] {pth_path}")

[저장] ./data/models/mission_16_mnist_cnn.pth


In [7]:
# 4-2. 동적 양자화 모델 저장
# quantize_dynamic은 CPU 전용 — MPS/CUDA 학습 후 반드시 CPU로 이동 필요
model_cpu = model.to('cpu')

# macOS(ARM/x86): qnnpack / Linux x86: fbgemm
# qnnpack은 크로스 플랫폼 호환되므로 기본값으로 고정
torch.backends.quantized.engine = 'qnnpack'

quantized_model = torch.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},       # FC 레이어만 양자화 (INT8)
    dtype=torch.qint8
)

quant_path = './data/models/mission_16_mnist_cnn_quantized.pth'
torch.save(quantized_model.state_dict(), quant_path)
print(f"[저장] {quant_path}")

[저장] ./data/models/mission_16_mnist_cnn_quantized.pth


/var/folders/l3/c5p364vd49z9t2353fr9w3nm0000gn/T/ipykernel_24755/3692671810.py:9: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [10]:
# 4-3. ONNX 변환 및 저장
import onnx

onnx_path = './data/models/mission_16_mnist_cnn.onnx'
dummy_input = torch.randn(1, 1, 28, 28)  # NCHW

model_cpu.eval()
torch.onnx.export(
    model_cpu,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=11
)

# ONNX 모델 유효성 검증
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"[저장] {onnx_path} (ONNX 검증 통과)")

/var/folders/l3/c5p364vd49z9t2353fr9w3nm0000gn/T/ipykernel_24755/2872022241.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0311 16:06:40.540000 24755 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0311 16:06:41.071000 24755 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale

[torch.onnx] Obtain model graph for `MnistCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MnistCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/opt/homebrew/Cellar/python@3.11/3.11.14_1/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/youuchul/Documents/github/01_deep_learning/16_Model-conversion/.venv/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/youuchul/Documents/github/01_deep_learning/16_Model-conversion/.venv/lib/python3.11/site-packages/onnxscript/versio

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.
[저장] ./data/models/mission_16_mnist_cnn.onnx (ONNX 검증 통과)


## 5. 모델 크기 비교

In [11]:
def get_file_size_kb(path: str) -> float:
    """파일 크기를 KB 단위로 반환."""
    return os.path.getsize(path) / 1024

sizes = {
    '.pth (일반)': get_file_size_kb(pth_path),
    '.pth (양자화)': get_file_size_kb(quant_path),
    '.onnx': get_file_size_kb(onnx_path),
}

print("=" * 50)
print("모델 크기 비교")
print("=" * 50)
for name, size_kb in sizes.items():
    print(f"{name:20s}: {size_kb:8.1f} KB")
print("=" * 50)

base = sizes['.pth (일반)']
for name, size_kb in sizes.items():
    ratio = size_kb / base * 100
    print(f"{name:20s}: {ratio:.1f}% of base")

모델 크기 비교
.pth (일반)           :   1650.7 KB
.pth (양자화)          :    472.5 KB
.onnx               :     14.2 KB
.pth (일반)           : 100.0% of base
.pth (양자화)          : 28.6% of base
.onnx               : 0.9% of base


In [12]:
# 최종 파일 목록 확인
import subprocess
result = subprocess.run(['ls', '-lh', './data/models/'], capture_output=True, text=True)
print(result.stdout)

total 7680
-rw-r--r--  1 youuchul  staff    14K Mar 11 16:06 mission_16_mnist_cnn.onnx
-rw-r--r--  1 youuchul  staff   1.7M Mar 11 16:06 mission_16_mnist_cnn.onnx.data
-rw-r--r--  1 youuchul  staff   1.6M Mar 11 16:02 mission_16_mnist_cnn.pth
-rw-r--r--  1 youuchul  staff   473K Mar 11 16:03 mission_16_mnist_cnn_quantized.pth

